[Guide Index](README.md)


# 01 Getting Started

StarLayer is a Python library for working with RDF 1.2, SPARQL 1.2 and SHACL 1.2, the current W3C drafts. Graphs are held by [Oxigraph](https://github.com/oxigraph/oxigraph), and validation is StarLayer's own SHACL engine.

This part of the guide installs StarLayer, builds a graph, queries it and validates it. Each section links to where the guide covers the topic in depth. It assumes a working knowledge of RDF, SPARQL and SHACL.

## How to run this notebook

Run the cells from top to bottom. Section 1 installs StarLayer if it is not installed, and the later sections reuse the graph built in section 2.

## 1. Install

The cell below installs StarLayer if it cannot be imported: from this repository when the notebook is run inside a checkout, otherwise from GitHub. After an install, restart the kernel and run the notebook again, because a package installed into a running kernel is not picked up. StarLayer follows the drafts as they change, so refresh the install now and then (run `pip install --upgrade` with the same URL, or `pip install -e packages/starlayer` again in a checkout). To work on StarLayer itself, see the repository [README](../../README.md).

In [1]:
# Install StarLayer if it cannot be imported (the import also checks that pyoxigraph is new enough).
try:
    import starlayer.graph  # noqa: F401
    print("starlayer is installed and current - skipping install.")
except ImportError as exc:
    import pathlib

    print(f"StarLayer needs installing or updating ({exc}).")
    checkout = pathlib.Path("../..").resolve()
    if (checkout / "packages" / "starlayer").is_dir():
        %pip install -e {checkout}/packages/starlayer
    else:
        %pip install --upgrade "git+https://github.com/hidden-graph/starlayer.git@shacl-engine#subdirectory=packages/starlayer"
    raise RuntimeError("StarLayer was installed: restart the kernel and run the notebook again.") from None

starlayer is installed and current - skipping install.


In [2]:
from starlayer.graph import Namespace, StarLayerGraph
from starlayer.shacl import validate

EX = "http://example.org/"
ex = Namespace(EX)

## 2. Using `StarLayerGraph`

`StarLayerGraph` holds a graph. `parse()` reads Turtle (and the other RDF formats, see [Serialization formats](06a-serialization-formats.ipynb)) and `serialize()` writes it; `add()`, `remove()` and `triples()` change and read it; `bind()` gives a prefix a name. A new graph has no prefixes, so bind the ones you need or pass `prefixes=` to the constructor.

Terms are pyoxigraph terms (`NamedNode`, `BlankNode`, `Literal` and `Triple`): `.value` is the text of an IRI or a literal. [Graphs](02-graphs.ipynb) covers what RDF 1.2 adds (triple terms, reification, direction-tagged literals), and [The Oxigraph-backed graph](02d-oxigraph-wrapper.ipynb) covers the class in full.

In [3]:
# Build a graph, and bind a prefix (a new graph has none).
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ;
      ex:name "Alice" ;
      ex:knows ex:bob .
""", format="turtle")

print("triples:", len(g))
# Oxigraph's own Turtle writer gives flat statements; serialize(..., pretty=True) groups and nests them.
print(g.serialize(format="turtle"))

triples: 3
@prefix ex: <http://example.org/> .
ex:alice ex:knows ex:bob ;
	ex:name "Alice" ;
	a ex:Person .



## 3. Querying

`query()` runs SPARQL and returns a `Result` that iterates over rows. A value in a row is a pyoxigraph term: `.value` is the bare IRI, and `str()` is its N-Triples form (`<http://...>`). See [SPARQL](03-sparql.ipynb) for the RDF 1.2 additions.

In [4]:
rows = g.query("""
    PREFIX ex: <http://example.org/>
    SELECT ?friend WHERE { ?person ex:knows ?friend }
""")
for row in rows:
    print(row.friend, "|", row.friend.value, "|", g.qname_term(row.friend))

<http://example.org/bob> | http://example.org/bob | ex:bob


## 4. Validating with SHACL

A shapes graph describes what data must look like. `validate(data_graph, shacl_graph=...)` checks a graph against it and returns a result with `conforms` and a readable `report_text`. See [SHACL shapes](04-shacl-shapes.ipynb) for the rest.

In [5]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")

result = validate(g, shacl_graph=shapes)
print("conforms:", result.conforms)

# Take away Alice's name: the shape needs at least one.
g.remove((ex.alice, ex.name, None))
result = validate(g, shacl_graph=shapes)
print("conforms:", result.conforms)
print(result.report_text)

conforms: True
conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:minCount Literal("1", datatype=<http://www.w3.org/2001/XMLSchema#integer>) ; sh:path ex:name ]
	Focus Node: ex:alice
	Result Path: ex:name
	Message: Needs at least 1 value for ex:name

